# 08 · Choosing the Distribution: Parametric Families, Heavy Tails & Count Data

Parametric probabilistic forecasting = *point model for the parameters* + *a distribution family*. The family is a
modelling decision as important as the mean model.

## Theory

### Parametric vs non-parametric
| | Parametric | Non-parametric |
|---|---|---|
| Representation | family + parameters, $Y\sim\mathcal D(\theta_t)$ | quantiles, samples, bins, conformal sets |
| Data needed | little | more |
| Tails / extrapolation beyond the data | determined by the family | limited by data |
| Risk | wrong family, so miscalibrated tails | crossing quantiles, sparse tails |

### A field guide to families
| Family | Support | Variance | Typical monthly use |
|---|---|---|---|
| Normal | $\mathbb R$ | $\sigma^2$ | aggregated, smooth series; log-scale models |
| Student-t | $\mathbb R$ | heavier tails ($\nu$ d.o.f.) | series with shocks / outliers |
| Log-normal | $(0,\infty)$ | $\propto\mu^2$ | multiplicative growth (sales, passengers) |
| Gamma | $(0,\infty)$ | $\phi\mu^2$ | positive skewed amounts |
| Poisson | $\{0,1,..\}$ | $=\mu$ | counts of independent events |
| **Negative Binomial** | $\{0,1,..\}$ | $\mu+\alpha\mu^2$ | **over-dispersed counts**: retail units, claims |
| Zero-inflated Poisson / NB | $\{0,1,..\}$ | extra zeros | sporadic demand |
| Tweedie ($1<p<2$) | $\{0\}\cup(0,\infty)$ | $\phi\mu^p$ | revenue with zeros, insurance (M5 winners used it) |
| Beta | $(0,1)$ | | rates, shares, occupancy |

NegBin arises as a **Gamma–Poisson mixture**: $\lambda\sim\text{Gamma}$, $Y|\lambda\sim\text{Poisson}(\lambda)$. That is
Poisson with a randomly varying rate, which is why real demand is almost always over-dispersed.

### Intermittent demand
Syntetos–Boylan classification by **ADI** (average inter-demand interval) and **CV²** of non-zero sizes:
smooth (ADI<1.32, CV²<0.49), erratic, intermittent, lumpy. Croston/SBA give *point* forecasts of the demand rate.
For **probabilistic** forecasts use count distributions (NegBin, zero-inflated, or bootstrapping), and evaluate
with scores valid for discrete distributions.

### Evaluating discrete forecasts
* Log score uses the **pmf**: $-\log\hat p(y)$
* CRPS for integer support: $\sum_{k\ge 0}\big(\hat F(k)-\mathbb 1\{y\le k\}\big)^2$
* Ordinary PIT is *not* uniform for discrete data. Use the **randomised PIT** $u=\hat F(y-1)+V\,\hat p(y)$.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(8)

## Part A: heavy tails. Normal vs Student-t residuals
We add four "promotion shocks" to retail Total (a common real-world pattern) and check which noise distribution fits
the SARIMA residuals better.

In [2]:
y = pf.load_retail_total().copy()
for d, f in [("2017-03-01", 1.18), ("2019-11-01", 0.85), ("2021-06-01", 1.22), ("2023-09-01", 1.15)]:
    y.loc[pd.Timestamp(d)] *= f
res = SARIMAX(np.log(y), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
e = np.asarray(res.resid)[13:]
e = (e - e.mean()) / e.std()

nu, loc_t, sc_t = stats.t.fit(e)
ll_norm = stats.norm.logpdf(e).sum()
ll_t = stats.t.logpdf(e, nu, loc_t, sc_t).sum()
print(f"Student-t fit: nu = {nu:.1f}")
print(f"log-likelihood  Normal = {ll_norm:.1f} (AIC {2*2 - 2*ll_norm:.1f})   Student-t = {ll_t:.1f} (AIC {2*3 - 2*ll_t:.1f})")

fig = pf.subplots(1, 2, titles=["Residual density (log y-axis shows the tails)", "QQ plot: points on the diagonal = good fit"], height=420)
grid = np.linspace(-5, 5, 400)
dens, edges = np.histogram(e, bins=30, density=True)
fig.add_trace(go.Bar(x=(edges[:-1] + edges[1:]) / 2, y=np.where(dens > 0, dens, np.nan), width=np.diff(edges) * 0.92,
                     name="residuals", marker_color=pf.INK["neutral"], hovertemplate="density %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=grid, y=stats.norm.pdf(grid), mode="lines", name="Normal", legendgroup="Normal",
                         line=dict(color=pf.SERIES[0], width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=grid, y=stats.t.pdf(grid, nu, loc_t, sc_t), mode="lines", name=f"Student-t (ν={nu:.1f})",
                         legendgroup="t", line=dict(color=pf.SERIES[1], width=2)), row=1, col=1)
fig.update_yaxes(type="log", range=[-3, 0], dtick=1, exponentformat="power", title_text="density", row=1, col=1)
pp = (np.arange(1, len(e) + 1) - 0.5) / len(e)
fig.add_trace(go.Scatter(x=[-4, 4], y=[-4, 4], mode="lines", name="y = x", line=dict(color=pf.INK["muted"], dash="dash", width=1),
                         hoverinfo="skip"), row=1, col=2)
fig.add_trace(go.Scatter(x=stats.norm.ppf(pp), y=np.sort(e), mode="markers", name="Normal quantiles", legendgroup="Normal",
                         showlegend=False, marker=dict(color=pf.SERIES[0], size=8, opacity=0.8)), row=1, col=2)
fig.add_trace(go.Scatter(x=stats.t.ppf(pp, nu, loc_t, sc_t), y=np.sort(e), mode="markers", name="Student-t quantiles",
                         legendgroup="t", showlegend=False, marker=dict(color=pf.SERIES[1], size=8, opacity=0.8)), row=1, col=2)
fig.update_xaxes(title_text="theoretical quantile", row=1, col=2)
fig.update_yaxes(title_text="residual quantile", row=1, col=2)
fig.show()

for q in [0.95, 0.99]:
    print(f"{q:.0%} quantile:  Normal {stats.norm.ppf(q):.2f}   t {stats.t.ppf(q, nu, loc_t, sc_t):.2f}   empirical {np.quantile(e, q):.2f}")

Student-t fit: nu = 4.5
log-likelihood  Normal = -151.8 (AIC 307.7)   Student-t = -149.2 (AIC 304.3)


95% quantile:  Normal 1.64   t 1.59   empirical 1.76
99% quantile:  Normal 2.33   t 2.72   empirical 2.73


A Gaussian assumption under-states the extreme quantiles when shocks exist, which matters for P99 safety stock or VaR.

## Part B: intermittent count demand for one SKU

In [3]:
sku = pf.make_sku_demand()
nz = sku[sku > 0]
adi = len(sku) / len(nz)
cv2 = (nz.std() / nz.mean()) ** 2
print(f"zeros: {np.mean(sku == 0):.0%}   ADI = {adi:.2f}   CV² = {cv2:.2f}   ->",
      "lumpy" if adi >= 1.32 and cv2 >= 0.49 else "intermittent" if adi >= 1.32 else "erratic" if cv2 >= 0.49 else "smooth")
print(f"mean = {sku.mean():.2f}, variance = {sku.var():.2f}  (Poisson would need variance ≈ mean)")

train, test = sku[:96], sku[96:]   # 8 years train, 5 years test
def design(idx):
    t = np.arange(len(sku))[sku.index.get_indexer(idx)]
    m = idx.month.to_numpy()
    return pd.DataFrame({"const": 1.0, "trend": t / 12, "sin": np.sin(2 * np.pi * m / 12), "cos": np.cos(2 * np.pi * m / 12)}, index=idx)
Xtr, Xte = design(train.index), design(test.index)
fig = go.Figure(go.Bar(x=sku.index, y=sku, name="units", marker_color=pf.SERIES[0], showlegend=False,
                       hovertemplate="%{x|%b %Y}: %{y:.0f} units<extra></extra>"))
pf.add_vline(fig, test.index[0], 0, sku.max(), "start of test period", color=pf.INK["primary"], dash="dash")
fig.update_layout(title="SKU monthly units", height=340, yaxis_title="units")
fig.show()

zeros: 31%   ADI = 1.44   CV² = 1.24   -> lumpy
mean = 2.49, variance = 13.86  (Poisson would need variance ≈ mean)


### Four candidate predictive distributions
1. **Gaussian** (OLS): continuous, allows negative demand.
2. **Poisson GLM** (log link): forces variance = mean.
3. **Negative Binomial**: estimates the over-dispersion α.
4. **Empirical climatology**: the histogram of past monthly demand.

In [4]:
ols = sm.OLS(train, Xtr).fit()
pois = sm.GLM(train, Xtr, family=sm.families.Poisson()).fit()
nb = NegativeBinomial(train, Xtr).fit(disp=0, maxiter=500)
alpha = float(np.asarray(nb.params)[-1])
print(f"Poisson Pearson dispersion = {pois.pearson_chi2 / pois.df_resid:.2f}  (≈1 if Poisson is right)")
print(f"NegBin alpha = {alpha:.2f}  -> Var = mu + {alpha:.2f}·mu²")

mu_g, sd_g = np.asarray(ols.predict(Xte)), np.sqrt(ols.scale)
mu_p = np.asarray(pois.predict(Xte))
mu_nb = np.asarray(nb.predict(Xte))
n_nb = 1 / alpha
emp = np.bincount(train.astype(int), minlength=200)[:200].astype(float)   # histogram of past demand on 0..199
emp[: int(train.max()) + 6] += 0.25                                       # light smoothing near the observed range only
emp = emp / emp.sum()

KMAX = np.arange(0, 200)
def pmf_cdf(name):
    """(N, KMAX) pmf and cdf matrices on the integer support for each test month."""
    if name == "Gaussian":           # discretise: P(k-0.5 < Y <= k+0.5), with all mass below 0.5 assigned to 0
        c = stats.norm.cdf(KMAX[None, :] + 0.5, mu_g[:, None], sd_g)
        pmf = np.diff(np.c_[np.zeros(len(mu_g)), c], axis=1)
    elif name == "Poisson":
        pmf = stats.poisson.pmf(KMAX[None, :], mu_p[:, None])
    elif name == "Negative Binomial":
        pmf = stats.nbinom.pmf(KMAX[None, :], n_nb, (n_nb / (n_nb + mu_nb))[:, None])
    else:
        pmf = np.tile(emp, (len(test), 1))
    return pmf, np.cumsum(pmf, axis=1)

print(f"Gaussian model: P(demand < 0) averaged over test months = {stats.norm.cdf(0, mu_g, sd_g).mean():.1%}  <- impossible values")

Poisson Pearson dispersion = 2.79  (≈1 if Poisson is right)
NegBin alpha = 0.92  -> Var = mu + 0.92·mu²
Gaussian model: P(demand < 0) averaged over test months = 14.8%  <- impossible values


In [5]:
yt = test.to_numpy().astype(int)
rows, pits = {}, {}
for name in ["Gaussian", "Poisson", "Negative Binomial", "Empirical"]:
    pmf, cdf = pmf_cdf(name)
    p_y = np.clip(pmf[np.arange(len(yt)), yt], 1e-12, None)
    crps = np.sum((cdf - (KMAX[None, :] >= yt[:, None])) ** 2, axis=1)
    lo = np.argmax(cdf >= 0.05, axis=1); hi = np.argmax(cdf >= 0.95, axis=1)
    F_ym1 = np.where(yt > 0, cdf[np.arange(len(yt)), np.maximum(yt - 1, 0)], 0.0)
    pits[name] = F_ym1 + rng.uniform(size=len(yt)) * p_y
    rows[name] = {"log score": np.mean(-np.log(p_y)), "CRPS": crps.mean(),
                  "90% coverage": np.mean((yt >= lo) & (yt <= hi)),
                  "pred. P(Y=0)": pmf[:, 0].mean(), "obs. share of zeros": np.mean(yt == 0)}
pd.DataFrame(rows).T

,log score,CRPS,90% coverage,pred. P(Y=0),obs. share of zeros
Gaussian,3.296,2.098,0.917,0.195,0.217
Poisson,3.299,2.191,0.767,0.104,0.217
Negative Binomial,2.324,2.028,0.950,0.249,0.217
Empirical,3.131,2.200,0.933,0.349,0.217


In [6]:
COLOR = {"Gaussian": pf.SERIES[0], "Poisson": pf.SERIES[1], "Negative Binomial": pf.SERIES[2], "Empirical": pf.SERIES[3]}
fig = pf.subplots(1, 4, titles=[f"randomised PIT: {name}" for name in pits], shared_yaxes=True, height=340)
for c, (name, u) in enumerate(pits.items(), start=1):
    pf.add_pit(fig, u, bins=5, row=1, col=c, color=COLOR[name])
fig.show()

i = int(np.argmax(mu_nb))
fig = go.Figure()
k = np.arange(0, 25)
top = 0
for name in ["Poisson", "Negative Binomial", "Gaussian"]:
    pmf, _ = pmf_cdf(name)
    top = max(top, pmf[i, :25].max())
    fig.add_trace(go.Bar(x=k, y=pmf[i, :25], name=name, marker_color=COLOR[name],
                         hovertemplate=name + ": P(Y=%{x}) = %{y:.3f}<extra></extra>"))
pf.add_vline(fig, yt[i], 0, top, "actual", color=pf.ACTUAL, dash="solid", width=2)
fig.update_layout(title=f"Predictive pmf for {test.index[i]:%b %Y}", barmode="group", height=400, xaxis_title="units",
                  yaxis_title="probability")
fig.show()

## Takeaways
* Check the **dispersion**: count data with variance ≫ mean needs NegBin (or a mixture), not Poisson.
  A Poisson model is over-confident: its intervals are too narrow, its PIT is U-shaped, and it under-predicts zeros.
* A Gaussian on low counts puts probability on **negative demand** and treats a discrete variable as continuous.
  It may still score acceptably on CRPS, but its quantiles and service levels are not meaningful at SKU level.
  It is fine for large aggregates.
* Heavy-tailed shocks call for Student-t (or empirical/bootstrap) errors, especially for high service levels.
* Always use **randomised PIT** and discrete CRPS / log score for counts.

### Exercises
1. Fit `statsmodels.discrete.count_model.ZeroInflatedNegativeBinomialP`. Does the extra zero component help here?
2. Implement Croston's method and turn it into a probabilistic forecast by assuming Poisson / NegBin demand at the Croston rate.
3. Aggregate the SKU to quarterly. How does the best family change? (Aggregation drives distributions towards Normal.)